# 04 Simulation: a simplified copy of SHOP's weekly problem (LP)

For each case, `lp_sim.py` solves a linear program over the 168 hours:

* **Objective:** sum over hours and plants of (price − water value of the plant's source reservoir) × production,
  minus the bypass/spill/river costs from the topology YAML, minus the value of water lost through bypasses and spillways
  (valued along the whole path to the sea).
* **Constraints:** hourly water balance of all 17 reservoirs, volume limits, plant capacities (sum of the units),
  and the minimum-flow and minimum-volume series (soft, with a high penalty).
* **Simplifications:** head fixed at the start level, efficiency 0.88, no on/off or start costs, no time delays,
  free tunnels between reservoirs with the same water value.

Solved with HiGHS (`scipy.optimize.linprog`) at three water-value scales (0.9, 1.0, 1.1), as in the teammate's report.
For each plant and hour it saves:

* `frac`: planned flow as a share of the plant's capacity;
* `margin`: price minus the LP's marginal value of the water, in EUR/MWh (> 0: worth running).

The 02 notebook uses these as features for the per-unit XGBoost. Run all cells top to bottom (about 10–15 minutes in total).
If the multiprocessing step fails, set `processes=1` (about 3–4 times slower).

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

import lp_sim as L      # the LP lives in lp_sim.py (same folder), so multiprocessing works in Jupyter on macOS

YAML = "data/extended/Tokke_Vinje_topology.yaml"
SCALES = (0.9, 1.0, 1.1)                                   # water values x 0.9, 1.0, 1.1
D = L.load_data("data")
M = L.build_model(YAML)
uc = pd.read_csv("data/kernel/Unit_commitment_decisions.csv", parse_dates=["starttime"], index_col="starttime")
test_cases = pd.read_csv("prediction_mapping.csv", parse_dates=["starttime"])
print(len(M["edges"]), "flows per hour,", M["n"], "variables per case")

t0 = time.time()
_, _, st = L.solve_case(M, *L.case_inputs(uc.index[:1], D, M["res"])[0], scales=SCALES)
print("One case, 3 scales: status", st, f"(0 = solved), {time.time() - t0:.2f}s")

56 flows per hour, 13776 variables per case
One case, 3 scales: status [0 0 0] (0 = solved), 0.24s


In [2]:
# All 2,922 training cases (about 5–10 minutes with 4 processes)
inputs = L.case_inputs(uc.index, D, M["res"])
frac, margin, status = L.run_cases(inputs, YAML, scales=SCALES, processes=4)
np.savez_compressed("lp_train.npz", frac=frac, margin=margin, status=status,
                    scales=np.array(SCALES), plants=np.array(L.PLANTS))
print(frac.shape, "| cases not solved:", (status != 0).any(axis=1).sum())   # (3, 2922, 168, 9)

200/2922 cases, 16s, not solved: 0
400/2922 cases, 30s, not solved: 0
600/2922 cases, 43s, not solved: 0
800/2922 cases, 56s, not solved: 0
1000/2922 cases, 72s, not solved: 0
1200/2922 cases, 86s, not solved: 0
1400/2922 cases, 101s, not solved: 0
1600/2922 cases, 115s, not solved: 0
1800/2922 cases, 131s, not solved: 0
2000/2922 cases, 145s, not solved: 0
2200/2922 cases, 160s, not solved: 0
2400/2922 cases, 175s, not solved: 0
2600/2922 cases, 189s, not solved: 0
2800/2922 cases, 204s, not solved: 0
2922/2922 cases, 212s, not solved: 0
(3, 2922, 168, 9) | cases not solved: 0


In [3]:
# The LP alone (no learning): how well does the plant margin rank the labels?
label_cols = [c for c in uc.columns if c.startswith("result_committed")]
units = list(dict.fromkeys(c.replace("result_committed_", "").rsplit("_t", 1)[0] for c in label_cols))
Y = uc[label_cols].to_numpy().reshape(len(uc), len(units), 168)
years = uc.index.year.to_numpy()
pi = [L.PLANTS.index(u.rsplit("_G", 1)[0]) for u in units]

rows = {}
for s_i, s in enumerate(SCALES):
    G = margin[s_i][:, :, pi].transpose(0, 2, 1)           # (case, unit, hour), like Y
    rows[f"scale {s}"] = {"all years": roc_auc_score(Y.ravel(), G.ravel()),
                          "2022": roc_auc_score(Y[years == 2022].ravel(), G[years == 2022].ravel()),
                          **{u: roc_auc_score(Y[:, k].ravel(), G[:, k].ravel()) for k, u in enumerate(units)}}
lp_alone = pd.DataFrame(rows)
lp_alone.round(3)

,scale 0.9,scale 1.0,scale 1.1
all years,0.944,0.967,0.966
2022,0.968,0.979,0.978
Hogga_G1,0.922,0.950,0.955
Tokke_G1,0.962,0.984,0.976
Tokke_G2,0.952,0.975,0.985
Tokke_G3,0.961,0.984,0.983
Tokke_G4,0.955,0.979,0.986
Lio_G1,0.934,0.939,0.929
Byrte_G1,0.978,0.994,0.990
Vinje_G1,0.934,0.976,0.983


In [4]:
# The 725 test cases (about 2–3 minutes)
test_inputs = L.case_inputs(test_cases["starttime"], D, M["res"])
frac_t, margin_t, status_t = L.run_cases(test_inputs, YAML, scales=SCALES, processes=4)
np.savez_compressed("lp_test.npz", frac=frac_t, margin=margin_t, status=status_t,
                    scales=np.array(SCALES), plants=np.array(L.PLANTS))
print(frac_t.shape, "| cases not solved:", (status_t != 0).any(axis=1).sum())   # (3, 725, 168, 9)

200/725 cases, 16s, not solved: 0
400/725 cases, 30s, not solved: 0
600/725 cases, 45s, not solved: 0
725/725 cases, 54s, not solved: 0
(3, 725, 168, 9) | cases not solved: 0
